In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

In [2]:
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

train = pd.read_csv(
    root / "data" / "raw" / "train.csv"
)

data = train.copy()
data["FamilySize"] = (
    data["SibSp"] + data["Parch"] + 1
)
data["IsAlone"] = (
    data["FamilySize"] == 1
).astype(int)

numerical_features = [
    "Age",
    "Fare",
    "FamilySize",
    "IsAlone",
]
categorical_features = [
    "Pclass",
    "Sex",
    "Embarked",
]

features = (
    numerical_features + categorical_features
)
X = data[features]
y = data["Survived"]

numerical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median"),
    ),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent"),
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ),
    ),
])

preprocessor = ColumnTransformer([
    (
        "numerical",
        numerical_pipeline,
        numerical_features,
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features,
    ),
])

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=1,
    ),
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [3]:
rows = []

for model_name, classifier in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "recall": "recall",
        },
        n_jobs=-1,
    )

    rows.append({
        "model": model_name,
        "accuracy_mean": scores[
            "test_accuracy"
        ].mean(),
        "accuracy_std": scores[
            "test_accuracy"
        ].std(),
        "survivor_recall_mean": scores[
            "test_recall"
        ].mean(),
    })

results = (
    pd.DataFrame(rows)
    .set_index("model")
    .sort_values(
        "accuracy_mean",
        ascending=False,
    )
)

display(results.round(3))

lr = results.loc["Logistic Regression"]
rf = results.loc["Random Forest"]
delta = (
    rf["accuracy_mean"]
    - lr["accuracy_mean"]
)

print(
    f"Accuracy delta (RF - LR): {delta:+.3f}"
)

,accuracy_mean,accuracy_std,survivor_recall_mean
model,,,
Random Forest,0.831,0.015,0.710
Logistic Regression,0.802,0.015,0.702


Accuracy delta (RF - LR): +0.028


## Model comparison decision

- Logistic Regression: accuracy **0.802 ± 0.015**, survivor recall **0.702**.
- Random Forest: accuracy **0.831 ± 0.015**, survivor recall **0.710**.
- Accuracy delta (Random Forest − Logistic Regression): **+0.028**.
- Decision: **Random Forest becomes the model-v2 candidate** because its accuracy improvement exceeds the `+0.005` rule. It also has slightly higher survivor recall with the same score variability.
- This is still a local validation result; Random Forest must beat the Kaggle baseline of **0.76794** before becoming the preferred competition model.

In [4]:
test = pd.read_csv(
    root / "data" / "raw" / "test.csv"
)

test_data = test.copy()
test_data["FamilySize"] = (
    test_data["SibSp"]
    + test_data["Parch"]
    + 1
)
test_data["IsAlone"] = (
    test_data["FamilySize"] == 1
).astype(int)

X_test = test_data[features]

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

rf_pipeline.fit(X, y)
rf_predictions = rf_pipeline.predict(X_test)

rf_submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": rf_predictions.astype(int),
})

assert rf_submission.shape == (418, 2)
assert list(rf_submission.columns) == [
    "PassengerId",
    "Survived",
]
assert rf_submission["PassengerId"].equals(
    test["PassengerId"]
)
assert set(
    rf_submission["Survived"].unique()
).issubset({0, 1})
assert not rf_submission.isna().any().any()

submission_dir = root / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)

rf_submission_path = (
    submission_dir / "random_forest_v2.csv"
)
rf_submission.to_csv(
    rf_submission_path,
    index=False,
)

logistic_submission = pd.read_csv(
    submission_dir / "logistic_family.csv"
)

assert logistic_submission[
    "PassengerId"
].equals(rf_submission["PassengerId"])

changed_predictions = (
    logistic_submission["Survived"]
    != rf_submission["Survived"]
).sum()

changed_rate = (
    changed_predictions / len(rf_submission)
)

predicted_not_survived = (
    rf_submission["Survived"] == 0
).sum()
predicted_survived = (
    rf_submission["Survived"] == 1
).sum()

In [5]:
from IPython.display import Markdown, display

display(Markdown(f"""
## Random Forest submission candidate

- Output file: `random_forest_v2.csv`
- Validated rows: **{len(rf_submission)}**
- Missing values: **{rf_submission.isna().sum().sum()}**
- Predicted not survived: **{predicted_not_survived}**
- Predicted survived: **{predicted_survived}**
- Predictions changed from `logistic-family-v1`: **{changed_predictions} of {len(rf_submission)} ({changed_rate:.1%})**
- Status: ready for Kaggle evaluation, but not submitted yet.
"""))


## Random Forest submission candidate

- Output file: `random_forest_v2.csv`
- Validated rows: **418**
- Missing values: **0**
- Predicted not survived: **274**
- Predicted survived: **144**
- Predictions changed from `logistic-family-v1`: **49 of 418 (11.7%)**
- Status: ready for Kaggle evaluation, but not submitted yet.
